# domain_qna RAGAS 평가

qna.parquet의 (question, answer)를 ground truth로 사용해
Qdrant Hybrid Search 검색 품질을 RAGAS 메트릭으로 정량 평가.

| 메트릭 | 의미 |
|---|---|
| `ContextRecall` | ground truth 답변이 검색된 컨텍스트로 커버되는 비율 |
| `ContextPrecisionWithReference` | 검색된 컨텍스트 중 실제 관련 있는 비율 |

## 0. 초기화

In [1]:
import os
import sys
sys.path.insert(0, "../../..")

import pandas as pd
from pathlib import Path
from tqdm import tqdm
from dotenv import load_dotenv
load_dotenv("../../../.env")

from qdrant_client import QdrantClient
from qdrant_client.models import (
    FieldCondition, Filter, Fusion, FusionQuery,
    MatchAny, MatchValue, Prefetch, SparseVector,
)
from fastembed import SparseTextEmbedding, TextEmbedding

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
client = QdrantClient(url=QDRANT_URL)

QNA_PATH = Path("../../../output/domain/qna.parquet")
df = pd.read_parquet(QNA_PATH)
print(f"qna.parquet: {len(df):,}행")
print(f"OPENAI_API_KEY: {'설정됨' if os.getenv('OPENAI_API_KEY') else '❌ 없음'}")
df.head(3)

/opt/anaconda3/envs/final-project/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


qna.parquet: 2,411행
OPENAI_API_KEY: 설정됨


,no,species,source,category,question,answer,notes
0,1,cat,bemypet,건강 및 질병,고양이 '췌장염'이 재발하기 쉬운 이유는 무엇인가요?,"고양이 췌장염은 원인이 불분명한 경우가 많고, 염증성 장질환(IBD)이나 간 질환이...",ISFM Guidelines; 췌장염은 완치보다 장기적인 관리가 중요한 질환임 =I...
1,2,cat,bemypet,사육 및 관리,췌장염을 앓았던 고양이의 식단 관리 원칙은?,"췌장에 무리를 주지 않도록 고단백, 저지방, 소화 흡수가 용이한 사료를 선택해야 합...",Hand et al. (2010); 처방식 사료 외의 기름진 간식 급여는 절대 금지...
2,3,cat,bemypet,건강 및 질병,고양이 '당뇨병'의 대표적인 초기 증상 '3다(多)'는?,"다음(물을 많이 마심), 다뇨(소변량이 늘어남), 다식(많이 먹지만 살은 빠짐)입니...",Nelson & Couto (2019); 갑작스러운 식욕 증가와 체중 감소가 동시에...


In [2]:
print("Dense 모델 로드 중...")
dense_model = TextEmbedding("intfloat/multilingual-e5-large")
print("Sparse 모델 로드 중...")
sparse_model = SparseTextEmbedding("Qdrant/bm25")
print("완료")

Dense 모델 로드 중...


/var/folders/n6/9fr973913k7931ymsg4ly6rc0000gn/T/ipykernel_44952/2718869934.py:2: UserWarning: The model intfloat/multilingual-e5-large now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  dense_model = TextEmbedding("intfloat/multilingual-e5-large")


Sparse 모델 로드 중...
완료


In [3]:
def embed(query: str):
    dv = list(dense_model.embed([f"query: {query}"]))[0].tolist()
    sv = list(sparse_model.embed([query]))[0]
    return dv, SparseVector(indices=sv.indices.tolist(), values=sv.values.tolist())


def search(query: str, top_k: int = 5, qdrant_filter=None) -> list[str]:
    """검색 후 (question + answer) 텍스트 리스트 반환 — RAGAS retrieved_contexts 형식"""
    dv, sv = embed(query)
    results = client.query_points(
        collection_name="domain_qna",
        prefetch=[
            Prefetch(query=dv, using="dense", limit=top_k * 3, filter=qdrant_filter),
            Prefetch(query=sv, using="sparse", limit=top_k * 3, filter=qdrant_filter),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=top_k,
        with_payload=True,
    ).points

    return [
        f"{p.payload.get('question', '')}\n{p.payload.get('answer', '')}".strip()
        for p in results
    ]

## 1. 평가 샘플 구성

species × category 층화 샘플링 (각 그룹 최대 3건, 총 ~50건)
RAGAS는 샘플당 LLM 호출이 발생하므로 작게 유지

In [4]:
parts = [
    g.sample(min(len(g), 3), random_state=42)
    for _, g in df.groupby(["species", "category"])
]
sample = pd.concat(parts).reset_index(drop=True)
print(f"샘플: {len(sample)}건")
sample.groupby(["species", "category"]).size()

샘플: 36건


species  category
both     건강 및 질병     3
         사육 및 관리     3
         여행 및 이동     3
         영양 및 식단     3
cat      건강 및 질병     3
         사육 및 관리     3
         영양 및 식단     3
         행동 및 심리     3
dog      건강 및 질병     3
         사육 및 관리     3
         영양 및 식단     3
         행동 및 심리     3
dtype: int64

## 2. Qdrant 검색 → RAGAS Dataset 구성

실제 LangGraph 조건: species + category 필터 적용

In [5]:
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset

ragas_samples = []

for _, row in tqdm(sample.iterrows(), total=len(sample), desc="검색 중"):
    sp = row["species"]
    must = [
        FieldCondition(
            key="species",
            match=MatchAny(any=[sp, "both"]) if sp != "both"
                  else MatchAny(any=["dog", "cat", "both"]),
        ),
        FieldCondition(key="category", match=MatchValue(value=row["category"])),
    ]
    contexts = search(row["question"], top_k=5, qdrant_filter=Filter(must=must))

    ragas_samples.append(SingleTurnSample(
        user_input=row["question"],
        retrieved_contexts=contexts,
        reference=row["answer"],
    ))

dataset = EvaluationDataset(samples=ragas_samples)
print(f"데이터셋 구성 완료: {len(dataset)}건")

검색 중: 100%|██████████| 36/36 [00:03<00:00, 11.71it/s]

데이터셋 구성 완료: 36건


## 3. RAGAS 평가

In [7]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextRecall, ContextPrecision

api_key = os.getenv("OPENAI_API_KEY")
openai_client = AsyncOpenAI(api_key=api_key)
llm = llm_factory("gpt-4o-mini", client=openai_client)

recall_scorer    = ContextRecall(llm=llm)
precision_scorer = ContextPrecision(llm=llm)

async def score_all():
    rows = []
    for s in tqdm(ragas_samples, desc="RAGAS 평가 중"):
        recall    = await recall_scorer.ascore(
            user_input=s.user_input,
            retrieved_contexts=s.retrieved_contexts,
            reference=s.reference,
        )
        precision = await precision_scorer.ascore(
            user_input=s.user_input,
            retrieved_contexts=s.retrieved_contexts,
            reference=s.reference,
        )
        rows.append({"question": s.user_input, "context_recall": recall.value, "context_precision": precision.value})
    return rows

results = asyncio.run(score_all())
df_result = pd.DataFrame(results)
print(f"\n=== 결과 요약 ===")
print(f"Context Recall:    {df_result['context_recall'].mean():.3f}")
print(f"Context Precision: {df_result['context_precision'].mean():.3f}")
df_result.describe()

RAGAS 평가 중: 100%|██████████| 36/36 [08:34<00:00, 14.30s/it]


=== 결과 요약 ===
Context Recall:    1.000
Context Precision: 0.923


,context_recall,context_precision
count,36.0,36.000000
mean,1.0,0.922994
std,0.0,0.107236
min,1.0,0.700000
25%,1.0,0.867014
50%,1.0,1.000000
75%,1.0,1.000000
max,1.0,1.000000


## 4. 결과 분석

In [11]:
display(df_result.describe())

,context_recall,context_precision
count,36.0,36.000000
mean,1.0,0.922994
std,0.0,0.107236
min,1.0,0.700000
25%,1.0,0.867014
50%,1.0,1.000000
75%,1.0,1.000000
max,1.0,1.000000


In [12]:
# recall 낮은 케이스 분석
low = df_result[df_result["context_recall"] < 0.5].copy()
print(f"Context Recall < 0.5: {len(low)}건")
for _, row in low.iterrows():
    print(f"\n[Q] {row['question'][:80]}")
    print(f"  recall={row['context_recall']:.2f}  precision={row['context_precision']:.2f}")

Context Recall < 0.5: 0건
